# 01 — Qdrant Intro: Collections, Points, Payloads

Qdrant e um banco de dados vetorial open-source escrito em Rust.
Neste notebook voce vai aprender as operacoes fundamentais.

**Prerequisito:** `docker compose up -d qdrant`

---

## Conceitos Fundamentais

```
Qdrant
└── Collection (como uma tabela)
    └── Point (como uma linha)
        ├── id: UUID ou int
        ├── vector: [0.1, 0.2, ...] (float array)
        └── payload: {"texto": "...", "categoria": "..."} (JSON)
```

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
    UpdateCollection, HnswConfigDiff,
)
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

# Conectar ao Qdrant
client = QdrantClient(host='localhost', port=6333)

# Verificar conexao
collections = client.get_collections()
print(f'Qdrant conectado!')
print(f'Collections existentes: {[c.name for c in collections.collections]}')

# Modelo de embedding
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dims
print(f'Modelo carregado: 384 dimensoes')

## 2.1 Criar uma Collection

In [ ]:
COLLECTION = 'artigos_tech'

# Recriar collection (limpa se ja existia)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=384,              # dimensoes do vetor
        distance=Distance.COSINE,  # metrica de distancia
    ),
)

info = client.get_collection(COLLECTION)
print(f'Collection criada: {COLLECTION}')
print(f'  Status: {info.status}')
print(f'  Dimensoes: {info.config.params.vectors.size}')
print(f'  Distancia: {info.config.params.vectors.distance}')
print(f'\nAcesse: http://localhost:6333/dashboard para ver no browser')

## 2.2 Inserir Points (documentos)

In [ ]:
# Nosso corpus de documentos
documentos = [
    {'id': 1, 'texto': 'Qdrant e um banco de dados vetorial open-source escrito em Rust.', 
     'categoria': 'banco_de_dados', 'ano': 2021},
    {'id': 2, 'texto': 'Embeddings sao representacoes densas de texto em espaco vetorial.', 
     'categoria': 'embeddings', 'ano': 2020},
    {'id': 3, 'texto': 'RAG combina recuperacao de documentos com geracao de linguagem.', 
     'categoria': 'rag', 'ano': 2023},
    {'id': 4, 'texto': 'HNSW e um algoritmo de indexacao para busca aproximada de vizinhos.', 
     'categoria': 'indexacao', 'ano': 2016},
    {'id': 5, 'texto': 'Sentence-transformers permite criar embeddings de frases facilmente.', 
     'categoria': 'embeddings', 'ano': 2019},
    {'id': 6, 'texto': 'Vetores de alta dimensao exigem algoritmos ANN para busca eficiente.', 
     'categoria': 'indexacao', 'ano': 2022},
    {'id': 7, 'texto': 'LangChain facilita a construcao de pipelines RAG complexos.', 
     'categoria': 'rag', 'ano': 2023},
    {'id': 8, 'texto': 'A quantizacao int8 reduz memoria de vetores em 4x com minima perda.', 
     'categoria': 'banco_de_dados', 'ano': 2023},
]

# Criar embeddings
textos = [d['texto'] for d in documentos]
embeddings = model.encode(textos, normalize_embeddings=True)

# Criar PointStructs
points = [
    PointStruct(
        id=doc['id'],
        vector=embeddings[i].tolist(),
        payload={
            'texto': doc['texto'],
            'categoria': doc['categoria'],
            'ano': doc['ano'],
        }
    )
    for i, doc in enumerate(documentos)
]

# Upsert (insert ou update)
result = client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f'Inseridos {len(points)} documentos')
print(f'Total de pontos na collection: {info.points_count}')

## 2.3 Busca Semantica

In [ ]:
def buscar(query, limit=3, filtro_categoria=None):
    query_vec = model.encode(query, normalize_embeddings=True)
    
    query_filter = None
    if filtro_categoria:
        query_filter = Filter(
            must=[FieldCondition(
                key='categoria',
                match=MatchValue(value=filtro_categoria)
            )]
        )
    
    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vec.tolist(),
        limit=limit,
        query_filter=query_filter,
        with_payload=True,
    ).points
    return results

# Teste 1: busca sem filtro
print('BUSCA: Como armazenar vetores eficientemente?')
print('-' * 50)
for r in buscar('Como armazenar vetores eficientemente?', limit=3):
    print(f'Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"]}')

print()

# Teste 2: busca com filtro de categoria
print('BUSCA (filtro: categoria=rag): Como construir pipeline?')
print('-' * 50)
for r in buscar('Como construir pipeline de perguntas e respostas?', limit=3, filtro_categoria='rag'):
    print(f'Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"]}')

## 2.4 Operacoes CRUD

In [ ]:
# READ: buscar um ponto por ID
pontos = client.retrieve(
    collection_name=COLLECTION,
    ids=[1, 3],
    with_payload=True,
    with_vectors=False,
)
print('Pontos recuperados por ID:')
for p in pontos:
    print(f'  ID {p.id}: {p.payload["texto"][:60]}')

# UPDATE: atualizar payload de um ponto
client.set_payload(
    collection_name=COLLECTION,
    payload={'atualizado': True, 'versao': 2},
    points=[1],
)
print('\nPayload atualizado no ponto 1')

# Verificar
p = client.retrieve(COLLECTION, ids=[1], with_payload=True)[0]
print(f'Payload do ponto 1: {p.payload}')

# DELETE: deletar um ponto
# client.delete(COLLECTION, points_selector=[ponto_id])  # descomentado = deleta!
print('\nDelete: demonstrado mas nao executado para preservar os dados')

## 2.5 Filtros Avancados com Payload

In [ ]:
from qdrant_client.models import (
    Filter, FieldCondition, MatchValue, Range
)

query_vec = model.encode('indexacao vetorial', normalize_embeddings=True)

# Filtro por intervalo de ano
resultado_filtrado = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=5,
    query_filter=Filter(
        must=[
            FieldCondition(
                key='ano',
                range=Range(gte=2022)  # ano >= 2022
            )
        ]
    ),
    with_payload=True,
).points

print('Busca: "indexacao vetorial" (filtro: ano >= 2022):')
for r in resultado_filtrado:
    print(f'  Score: {r.score:.3f} | Ano: {r.payload["ano"]} | {r.payload["texto"][:60]}')

print()

# Filtro NOT (excluir categoria)
from qdrant_client.models import Filter as QFilter
resultado_sem_rag = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=5,
    query_filter=Filter(
        must_not=[
            FieldCondition(key='categoria', match=MatchValue(value='rag'))
        ]
    ),
    with_payload=True,
).points

print('Busca: "indexacao vetorial" (excluindo categoria=rag):')
for r in resultado_sem_rag:
    print(f'  Score: {r.score:.3f} | [{r.payload["categoria"]}] {r.payload["texto"][:60]}')

## 2.6 Scroll (paginar todos os pontos)

In [ ]:
# Listar todos os pontos sem query (scroll)
all_points, next_cursor = client.scroll(
    collection_name=COLLECTION,
    limit=10,
    with_payload=True,
    with_vectors=False,
)

print(f'Total de pontos na collection: {len(all_points)}')
print('\nTodos os documentos indexados:')
for p in all_points:
    print(f'  ID {p.id} | [{p.payload["categoria"]}] {p.payload["texto"][:60]}')

## Resumo

| Operacao | Metodo Qdrant | Uso |
|---------|--------------|-----|
| Criar collection | `create_collection()` | Inicializar espaco vetorial |
| Inserir/atualizar | `upsert()` | Adicionar documentos |
| Busca semantica | `search()` | Encontrar similares |
| Busca com filtro | `search(query_filter=...)` | Combinar semantica + metadata |
| Recuperar por ID | `retrieve()` | Busca exata por ID |
| Listar tudo | `scroll()` | Paginar collection |
| Deletar | `delete()` | Remover pontos |

## Proximo
- [02 — HNSW Indexing](02_hnsw_indexing.html): Como o indice funciona por dentro